In [7]:
import numpy as np
import pandas as pd
df2=pd.read_csv("../delhicar.csv")
print(df2["variant"].nunique())
print(df2.filter(like='variant').shape)


1343
(8137, 1)


In [9]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df2=pd.read_csv("../delhicar.csv")

# remove curse of dimensionality form variants by grouping rare varinats udner other
valuecounts=df2["variant"].value_counts()
rarelements=valuecounts[valuecounts<15].index
df2["variant"]=df2["variant"].replace(rarelements,"Other")
df2=df2.drop("latest_publish_date",axis=1)
df2=pd.get_dummies(df2,drop_first=True)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
x=df2.drop("price",axis=1)
y=df2[["price"]]
model=LinearRegression()
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
model.fit(X_train,y_train)
prediction=model.predict(X_test)
r2score=r2_score(y_test,prediction)


train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Train R²:", r2_score(y_train, train_pred))
print("Test R²:", r2_score(y_test, test_pred))

print(r2score)


Train R²: 0.9080644739692731
Test R²: 0.870255142259133
0.870255142259133


In [13]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import  r2_score,mean_absolute_error
import numpy as np

le = LabelEncoder()
df2 = pd.read_csv("../delhicar.csv")

# remove curse of dimensionality from variants by grouping rare variants under "Other"
valuecounts = df2["variant"].value_counts()
rarelements = valuecounts[valuecounts < 15].index
df2["variant"] = df2["variant"].replace(rarelements, "Other")

print("Variant categories after grouping:", df2["variant"].nunique())   # ← INSERT HERE

df2 = df2.drop("latest_publish_date", axis=1)
df2 = pd.get_dummies(df2, drop_first=True)   # variant becomes variant_XXX columns from here onward

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
df2["price"]=np.log1p(df2["price"])
x = df2.drop("price", axis=1)
y = df2["price"]

model = LinearRegression()
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)
print("Train R²:", r2_score(y_train, train_pred))
print("Test R²:", r2_score(y_test, test_pred))

Variant categories after grouping: 127
Train R²: 0.969384118369973
Test R²: 0.9606493575585063


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,OneHotEncoder,FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
linearmodeldf = pd.read_csv("../car.csv")
print(linearmodeldf.columns.tolist())



X = linearmodeldf.drop("selling_price", axis=1)
y = linearmodeldf["selling_price"]
# log transform target
y_log = np.log1p(y)
X_train, X_test, y_train_log, y_test_log = train_test_split(
X, y_log, test_size=0.2, random_state=42
)
tf1=ColumnTransformer([
    ("Onehotencoder",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),["car_name","brand","model","fuel_type","seller_type","transmission_type"]),
    ("Logtransformer",FunctionTransformer(np.log1p),["km_driven"]),
    ("StandardScaler",StandardScaler(),["vehicle_age","engine","max_power","mileage","seats"]),
])
pipeline=Pipeline(
    [
        ("preprocessor",tf1),
        (
         "model",LinearRegression()
        )
    ]
)


pipeline.fit(X_train, y_train_log)
# predict log price

# testing model with custom input

# custom_input = pd.DataFrame({
#     "car_name": ["Honda City"],
#     "brand": ["Honda"],
#     "model": ["City"],
#     "vehicle_age": [5],
#     "km_driven": [45000],
#     "seller_type": ["Dealer"],
#     "fuel_type": ["Petrol"],
#     "engine": [1498],
#     "mileage": [17.8],
#     "max_power": [117.3]
# })

custom_input = pd.DataFrame({
    "car_name": ["Mercedes-Benz C-Class"],
    "brand": ["Mercedes-Benz"],
    "model": ["C-Class"],
    "vehicle_age": [4],
    "km_driven": [38000],
    "seller_type": ["Dealer"],
    "fuel_type": ["Diesel"],
    "engine": [1950],
    "mileage": [19.0],
    "max_power": [191.0],
    "seats": [6],
    "transmission_type": ["Automatic"],
})

pred_log = pipeline.predict(custom_input)
pred_price = np.expm1(pred_log)

print(f"Predicted Price: ₹{pred_price[0]:,.0f}")

pred_log = pipeline.predict(X_test)
# convert back to actual price
pred_price = np.expm1(pred_log)
y_test_price = np.expm1(y_test_log)
print("R2 Score:", round(r2_score(y_test_price, pred_price), 4))
print("MAE:", round(mean_absolute_error(y_test_price, pred_price), 2))



['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']
Predicted Price: ₹2,452,329
R2 Score: 0.9046
MAE: 116370.97


In [3]:
linearmodeldf.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
